# TA-003 — Entrenamiento ResNet-50 para Benchmarking CNN
**Proyecto:** Sistema Multimodal de IA para Apoyo Diagnóstico Clínico Veterinario — Vargas Vet  
**Curso:** Taller Integrador 1 — UPAO  
**Integrantes:** Baylón Toledo, Diogho Matteo (PM) · Saavedra Arroyo, Sebastián Alonso (Scrum Master)  
**Sprint 2:** 31 May – 20 Jun 2026  

**Objetivo del notebook:**
1. Verificar que PyTorch usa GPU (RTX 4060)
2. Cargar manifests de train/val/test desde TA-001
3. Implementar Dataset personalizado para radiografías DICOM
4. Calcular pesos de clases para balanceo
5. Configurar ResNet-50 pretrained con cabeza personalizada
6. Ejecutar FASE 1 (feature extraction 5 épocas)
7. Ejecutar FASE 2 (fine-tuning completo con early stopping)
8. Graficar curvas de entrenamiento
9. Evaluar en test set y medir tiempo de inferencia
10. Guardar métricas para DO-002 (reporte benchmarking)

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print('Compute capability:', torch.cuda.get_device_properties(0).major, '.', torch.cuda.get_device_properties(0).minor, sep='')
    DEVICE = 'cuda'
else:
    print('ADVERTENCIA: GPU no detectada. Verificar instalacion de CUDA y PyTorch+cu130')
    DEVICE = 'cpu'

print('Dispositivo activo:', DEVICE)

## 1 · Verificación de entorno GPU
> Antes de cualquier operación se verifica que PyTorch detecte la RTX 4060. Si el resultado muestra `cuda` y el nombre de la GPU, el entorno está correcto para el entrenamiento.

## 2 · Imports y configuración

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve, auc
from pathlib import Path
import time
import json
from tqdm import tqdm
from PIL import Image

sns.set_style('whitegrid')

BASE = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\Radiografias')
MANIFESTS = BASE / 'dataset_split' / 'manifests'
DATASET_SPLIT = BASE / 'dataset_split'
CHECKPOINTS_DIR = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\checkpoints')
CHECKPOINTS_DIR.mkdir(exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
CLASSES = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
EXCLUDE_TAGS = {'fracture', 'pneumomediastinum', 'costal_fracture', 'exclude'}

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Configuración:')
print(f'  Device: {DEVICE}')
print(f'  Clases: {NUM_CLASSES}')
print(f'  Tamaño imagen: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'  Seed: {SEED}')

## 3 · Cargar y explorar manifests

In [ ]:
df_train = pd.read_csv(MANIFESTS / 'train.csv')
df_val = pd.read_csv(MANIFESTS / 'val.csv')
df_test = pd.read_csv(MANIFESTS / 'test.csv')

print('Dataset splits:')
print(f'  Train: {len(df_train)} imágenes | {df_train["PatientName"].nunique()} pacientes')
print(f'  Val:   {len(df_val)} imágenes | {df_val["PatientName"].nunique()} pacientes')
print(f'  Test:  {len(df_test)} imágenes | {df_test["PatientName"].nunique()} pacientes')
print()
print('Distribución de clases en train:')
for cls in CLASSES:
    mask = df_train['TAG'].str.contains(cls, na=False)
    count = mask.sum()
    pct = count / len(df_train) * 100
    print(f'  {cls}: {count} ({pct:.1f}%)')

## 4 · Clase VetXRayDataset
> Lee archivos DICOM, normaliza con percentil 2-98, maneja MONOCHROME1, replica canal gris a pseudo-RGB, y aplica transforms.

In [ ]:
def load_dicom_normalized(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    p2, p98 = np.percentile(img, [2, 98])
    img = np.clip(img, p2, p98)
    img = (img - p2) / (p98 - p2 + 1e-8)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        img = 1.0 - img
    return (img * 255).astype(np.uint8)

def build_label_vector(tag_str, classes):
    vec = np.zeros(len(classes), dtype=np.float32)
    if pd.isna(tag_str):
        vec[-1] = 1.0
        return vec
    tags = [t.strip() for t in str(tag_str).split('|')]
    for i, cls in enumerate(classes):
        if cls in tags:
            vec[i] = 1.0
    if vec.sum() == 0:
        vec[-1] = 1.0
    return vec

class VetXRayDataset(Dataset):
    def __init__(self, df, base_path, classes, split='train'):
        self.df = df.reset_index(drop=True)
        self.base_path = Path(base_path)
        self.classes = classes
        self.split = split

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dcm_path = self.base_path / self.split / row['FileName']
        img_gray = load_dicom_normalized(dcm_path)
        img_pil = Image.fromarray(img_gray)
        img_rgb = Image.merge('RGB', [img_pil, img_pil, img_pil])
        img_tensor = self.transform(img_rgb)
        label = build_label_vector(row['TAG'], self.classes)
        return img_tensor, torch.from_numpy(label)

print('VetXRayDataset clase definida')

## 5 · Verificar dataset

In [ ]:
train_ds = VetXRayDataset(df_train, DATASET_SPLIT, CLASSES, split='train')
val_ds = VetXRayDataset(df_val, DATASET_SPLIT, CLASSES, split='val')
test_ds = VetXRayDataset(df_test, DATASET_SPLIT, CLASSES, split='test')

print(f'Dataset sizes: train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}')

img, lbl = train_ds[0]
print(f'Forma tensor: {img.shape}')
print(f'Forma label: {lbl.shape}')
print(f'Label ejemplo: {lbl.numpy()}')
print(f'Clases con valor 1: {[CLASSES[i] for i in range(len(CLASSES)) if lbl[i]==1]}')

## 6 · Visualizar muestra del dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (ds, split) in zip(axes, [(train_ds, 'train'), (val_ds, 'val'), (test_ds, 'test')]):
    idx = np.random.randint(0, len(ds))
    img, lbl = ds[idx]
    img_denorm = img.numpy().transpose(1, 2, 0)
    img_denorm = (img_denorm * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    ax.imshow(img_denorm)
    ax.axis('off')
    tags = [CLASSES[i] for i in range(len(CLASSES)) if lbl[i]==1]
    ax.set_title(f'[{split.upper()}]\nTAGs: {" | ".join(tags[:2])}', fontsize=9)

fig.suptitle('Muestra visual — una radiografía por split', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7 · Calcular class weights

In [ ]:
def compute_class_weights(df, classes):
    N = len(df)
    weights = np.zeros(len(classes))
    for i, cls in enumerate(classes):
        n_pos = df['TAG'].str.contains(cls, na=False).sum()
        n_neg = N - n_pos
        if n_pos > 0:
            weights[i] = n_neg / n_pos
        else:
            weights[i] = 1.0
    return weights

class_weights = compute_class_weights(df_train, CLASSES)
class_weights_tensor = torch.from_numpy(class_weights).float().to(DEVICE)

print('Class weights (para BCEWithLogitsLoss):')
for cls, w in zip(CLASSES, class_weights):
    print(f'  {cls}: {w:.4f}')

## 8 · Construir modelo ResNet-50

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'ResNet-50 arquitectura:')
print(f'  Parámetros totales: {total_params:,}')
print(f'  Parámetros entrenables inicialmente: {trainable_params:,}')
print(f'  Clases de salida: {NUM_CLASSES}')

## 9 · Funciones de entrenamiento y evaluación

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.best_epoch = None
    
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    loss_list = []
    for imgs, lbls in tqdm(loader, desc='Train', leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_list.append(loss.item())
    return np.mean(loss_list)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_list = []
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Eval', leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss_list.append(loss.item())
            all_logits.append(logits.cpu().numpy())
            all_labels.append(lbls.cpu().numpy())
    val_loss = np.mean(loss_list)
    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))
    val_auc = roc_auc_score(all_labels, probs, average='macro')
    val_f1 = f1_score(all_labels, (probs > 0.5).astype(int), average='macro', zero_division=0)
    val_acc = accuracy_score(all_labels.flatten(), (probs > 0.5).astype(int).flatten())
    return val_loss, val_auc, val_f1, val_acc

print('Funciones de entrenamiento y evaluación definidas')

## 10 · Configuración y dataloaders

In [ ]:
EPOCHS_MAX = 50
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
PHASE1_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 10

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Configuración de entrenamiento:')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Learning rate: {LR}')
print(f'  Weight decay: {WEIGHT_DECAY}')
print(f'  FASE 1 (feature extraction): {PHASE1_EPOCHS} épocas')
print(f'  FASE 2 (fine-tuning): hasta {EPOCHS_MAX - PHASE1_EPOCHS} épocas')
print(f'  Early stopping patience: {EARLY_STOPPING_PATIENCE}')

## 11 · FASE 1 — Feature Extraction (5 épocas)
> Se congela el backbone (ResNet-50) y se entrena solo la cabeza. Esto permite que el modelo se adapte al dominio médico sin perder los features de ImageNet.

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor)
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda')

history_phase1 = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': [], 'val_acc': []}

print('\n=== FASE 1: Feature Extraction (5 épocas) ===')
for epoch in range(PHASE1_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_auc, val_f1, val_acc = eval_epoch(model, val_loader, criterion, DEVICE)
    
    history_phase1['train_loss'].append(train_loss)
    history_phase1['val_loss'].append(val_loss)
    history_phase1['val_auc'].append(val_auc)
    history_phase1['val_f1'].append(val_f1)
    history_phase1['val_acc'].append(val_acc)
    
    print(f'Época {epoch+1}/{PHASE1_EPOCHS} | TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f} | Acc: {val_acc:.4f}')

## 12 · FASE 2 — Fine-tuning Completo
> Se descongelan todos los parámetros. El learning rate para capas tempranas es menor para preservar features ImageNet. Se usa ReduceLROnPlateau y early stopping.

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR/10, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
scaler = torch.amp.GradScaler('cuda')

best_val_auc = 0
best_epoch = 0
best_checkpoint_path = CHECKPOINTS_DIR / 'resnet50_best.pth'

history_phase2 = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': [], 'val_acc': []}

print('\n=== FASE 2: Fine-tuning Completo ===')
for epoch in range(EPOCHS_MAX - PHASE1_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_auc, val_f1, val_acc = eval_epoch(model, val_loader, criterion, DEVICE)
    
    history_phase2['train_loss'].append(train_loss)
    history_phase2['val_loss'].append(val_loss)
    history_phase2['val_auc'].append(val_auc)
    history_phase2['val_f1'].append(val_f1)
    history_phase2['val_acc'].append(val_acc)
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = PHASE1_EPOCHS + epoch + 1
        torch.save(model.state_dict(), best_checkpoint_path)
    
    scheduler.step(val_auc)
    should_stop = early_stopping(val_auc)
    
    print(f'Época {PHASE1_EPOCHS + epoch + 1}/{EPOCHS_MAX} | TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f} | Acc: {val_acc:.4f}')
    
    if should_stop:
        print(f'\nEarly stopping activado en época {PHASE1_EPOCHS + epoch + 1}')
        break

total_epochs_trained = PHASE1_EPOCHS + len(history_phase2['train_loss'])
print(f'\nMejor épocas: {best_epoch} con AUC validación: {best_val_auc:.4f}')

## 13 · Curvas de entrenamiento

In [ ]:
all_epochs = list(range(1, total_epochs_trained + 1))
all_train_loss = history_phase1['train_loss'] + history_phase2['train_loss']
all_val_loss = history_phase1['val_loss'] + history_phase2['val_loss']
all_val_auc = history_phase1['val_auc'] + history_phase2['val_auc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(all_epochs, all_train_loss, 'o-', label='Train Loss', alpha=0.7)
axes[0].plot(all_epochs, all_val_loss, 's-', label='Val Loss', alpha=0.7)
axes[0].axvline(x=PHASE1_EPOCHS + 0.5, color='red', linestyle='--', alpha=0.5, label='Fine-tuning start')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss durante el entrenamiento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_epochs, all_val_auc, 'o-', color='green', alpha=0.7, label='Val AUC')
axes[1].axhline(y=0.75, color='orange', linestyle='--', alpha=0.7, label='Umbral TA-001 (0.75)')
axes[1].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best ({best_epoch})')
axes[1].axvline(x=PHASE1_EPOCHS + 0.5, color='purple', linestyle='--', alpha=0.5, label='Fine-tuning start')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('AUC (macro)')
axes[1].set_title('AUC en validación')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Curvas de entrenamiento — ResNet-50', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 14 · Evaluación en Test Set

In [ ]:
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
model.eval()

test_loss, test_auc, test_f1, test_acc = eval_epoch(model, test_loader, criterion, DEVICE)

print(f'\n=== Resultados en TEST SET ===')
print(f'  AUC (macro): {test_auc:.4f}')
print(f'  F1-score (macro): {test_f1:.4f}')
print(f'  Accuracy: {test_acc:.4f}')
print(f'  Loss: {test_loss:.4f}')
print(f'  Cumple umbral AUC >= 0.75: {test_auc >= 0.75}')
print(f'  Cumple umbral F1 >= 0.75: {test_f1 >= 0.75}')

print('\n=== Tiempo de Inferencia ===')
times = []
with torch.no_grad():
    for i, (imgs, _) in enumerate(test_loader):
        if i >= 50 // BATCH_SIZE:
            break
        imgs = imgs.to(DEVICE)
        t0 = time.time()
        _ = model(imgs)
        t1 = time.time()
        times.append((t1 - t0) / BATCH_SIZE)

inference_time_per_image = np.mean(times)
print(f'  Tiempo promedio por imagen: {inference_time_per_image:.4f} seg')
print(f'  Cumple umbral < 6 seg: {inference_time_per_image < 6}')

## 15 · Tabla de Resultados Finales

In [ ]:
results_df = pd.DataFrame({
    'Métrica': [
        'AUC (validación)',
        'AUC (test)',
        'F1-macro (test)',
        'Accuracy (test)',
        'Épocas entrenadas',
        'Mejor época',
        'Inferencia (seg/img)',
        'Cumple AUC >= 0.75',
        'Cumple F1 >= 0.75',
        'Cumple Inferencia < 6 seg'
    ],
    'ResNet-50': [
        f'{best_val_auc:.4f}',
        f'{test_auc:.4f}',
        f'{test_f1:.4f}',
        f'{test_acc:.4f}',
        f'{total_epochs_trained}',
        f'{best_epoch}',
        f'{inference_time_per_image:.4f}',
        'Sí' if test_auc >= 0.75 else 'No',
        'Sí' if test_f1 >= 0.75 else 'No',
        'Sí' if inference_time_per_image < 6 else 'No'
    ]
})

print('\n')
print(results_df.to_string(index=False))

## 16 · Guardar resultados en JSON

In [ ]:
results_json = {
    'model': 'resnet50',
    'val_auc': float(best_val_auc),
    'test_auc': float(test_auc),
    'test_f1_macro': float(test_f1),
    'test_accuracy': float(test_acc),
    'epochs_trained': int(total_epochs_trained),
    'best_epoch': int(best_epoch),
    'inference_time_sec': float(inference_time_per_image),
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'weight_decay': WEIGHT_DECAY,
    'phase1_epochs': PHASE1_EPOCHS,
    'num_classes': NUM_CLASSES,
    'dataset_split': 'train: 4914 | val: 1063 | test: 940'
}

results_json_path = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\Notebooks\resnet50_results.json')
with open(results_json_path, 'w') as f:
    json.dump(results_json, f, indent=4)

print(f'Resultados guardados en: {results_json_path}')
print(f'Checkpoint del mejor modelo: {best_checkpoint_path}')

---
## Resumen Final

| Criterio TA-003 | Estado |
|---|---|
| ResNet-50 entrenado con transfer learning | Completado |
| FASE 1 (feature extraction 5 épocas) | Completado |
| FASE 2 (fine-tuning + early stopping) | Completado |
| Mixed precision (torch.cuda.amp) | Activado |
| BCEWithLogitsLoss + class weights | Aplicado |
| Métricas registradas para DO-002 | JSON + tabla |
| Checkpoint del mejor modelo guardado | resnet50_best.pth |

**Próximo paso:** TA-004 — Entrenar EfficientNet-B0 y DenseNet-121, comparar resultados.